In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Public Schools Dashboard — Cambodia (2018–2021)
Mirrors the structure of `private_dashboard.ipynb`.

## 1. Schools, Classes & Staff by Province

In [2]:
pub_cols = [
    'Province', 'Schools', 'Disadv_Schools', 'Classes', 'Classes_Pagoda',
    'Enrollment_Total', 'Enrollment_Girl',
    'Repeaters_Total', 'Repeaters_Girl',
    'TeachStaff_Total', 'TeachStaff_Female',
    'NonTeachStaff_Total', 'NonTeachStaff_Female',
    'TotalStaff_Total', 'TotalStaff_Female'
]

def clean_pub_main(df):
    # Rows 0-2 are always: title, header-row-1, header-row-2 — skip all three
    df = df.iloc[3:].copy()
    df.columns = pub_cols
    df = df[df['Province'].notna()].reset_index(drop=True)
    df = df.set_index('Province')
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df

frames = []
for year, fname in [
    (2018, 'organized_education_data/public_schools_2018-2019.xlsx'),
    (2019, 'organized_education_data/public_schools_2019-2020.xlsx'),
    (2020, 'organized_education_data/public_schools_2020-2021.xlsx'),
]:
    raw = pd.read_excel(fname, sheet_name='Schools, Classes & Staff by Pro', header=None)
    df  = clean_pub_main(raw)
    df['Year'] = year
    frames.append(df)

combined_pub_main = pd.concat(frames)
cols = list(combined_pub_main.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_main = combined_pub_main[cols]

print('Shape:', combined_pub_main.shape)
combined_pub_main.head()

Shape: (84, 15)


,Year,Schools,Disadv_Schools,Classes,Classes_Pagoda,Enrollment_Total,Enrollment_Girl,Repeaters_Total,Repeaters_Girl,TeachStaff_Total,TeachStaff_Female,NonTeachStaff_Total,NonTeachStaff_Female,TotalStaff_Total,TotalStaff_Female
Province,,,,,,,,,,,,,,,
Banteay Meanchey,2018,818,15,4598,11,152590,76099,4051,1371,4782,2440,888,210,5670,2650
Battambang,2018,1133,17,7494,17,253481,125579,12244,4362,6946,4165,1811,672,8757,4837
Kampong Cham,2018,804,27,5846,111,221707,110363,14544,5235,5835,3390,1695,659,7530,4049
Kampong Chhnang,2018,466,0,3220,7,118144,59641,5080,1676,3498,1682,700,188,4198,1870
Kampong Speu,2018,603,2,4162,11,170222,83971,5931,2157,4471,1957,742,129,5213,2086


## 2. Enrollment & Repeaters by Grade

- **2018-19**: four separate 13-column sheets (grades 1-3, 4-6, 7-9, 10-12) — merged here
- **2019-20 & 2020-21**: single 49-column sheet (Province + 12 grades × 4 sub-cols)

Total columns per grade: `Enrollment_Total`, `Enrollment_Girl`, `Repeaters_Total`, `Repeaters_Girl`.  
Row totals are **computed** for all years (the public sheets do not include a pre-built total column).

In [3]:
sub_hdrs = ['Enrollment_Total', 'Enrollment_Girl', 'Repeaters_Total', 'Repeaters_Girl']

# 49-column layout: Province + Grade_1..12 × 4 sub-cols (NO pre-built totals)
enroll_cols_pub = ['Province']
for i in range(1, 13):
    for s in sub_hdrs:
        enroll_cols_pub.append(f'Grade_{i}_{s}')


def _find_data_start(df):
    """Return row index of first real province (skips title + header rows)."""
    return next(
        i for i, v in enumerate(df.iloc[:, 0])
        if isinstance(v, str) and v not in ('Province',) and not v.startswith('Table')
    )


def parse_enroll_block(df, g_start):
    """13-column split sheet: Province + 3 grades × 4 sub-cols."""
    start = _find_data_start(df)
    data  = df.iloc[start:].copy().reset_index(drop=True)
    data  = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    block_cols = ['Province']
    for g in range(g_start, g_start + 3):
        for s in sub_hdrs:
            block_cols.append(f'Grade_{g}_{s}')
    data.columns = block_cols
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce').astype('Int64')


def parse_enroll_single(df):
    """49-column sheet: Province + 12 grades × 4 sub-cols."""
    start = _find_data_start(df)
    data  = df.iloc[start:].copy().reset_index(drop=True)
    data  = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    data.columns = enroll_cols_pub
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce').astype('Int64')


def add_totals(df):
    """Compute and append Total_* columns (sum across all 12 grades)."""
    for s in sub_hdrs:
        df[f'Total_{s}'] = df[[f'Grade_{g}_{s}' for g in range(1, 13)]].sum(axis=1)
    return df


# ── 2018-19: four split sheets ───────────────────────────────────────────────
f18 = 'organized_education_data/public_schools_2018-2019.xlsx'
enroll_18 = add_totals(pd.concat([
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters by Grad', header=None), 1),
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters (Gr1-4)', header=None), 4),
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters (Gr5-8)', header=None), 7),
    parse_enroll_block(pd.read_excel(f18, sheet_name='Enrollment & Repeaters(Gr9-12)', header=None), 10),
], axis=1))
enroll_18['Year'] = 2018

# ── 2019-20 & 2020-21: single sheet ─────────────────────────────────────────
enroll_19 = add_totals(parse_enroll_single(
    pd.read_excel('organized_education_data/public_schools_2019-2020.xlsx',
                  sheet_name='Enrollment & Repeaters by Grade', header=None)))
enroll_19['Year'] = 2019

enroll_20 = add_totals(parse_enroll_single(
    pd.read_excel('organized_education_data/public_schools_2020-2021.xlsx',
                  sheet_name='Enrollment & Repeaters by Grade', header=None)))
enroll_20['Year'] = 2020

# ── Combine ──────────────────────────────────────────────────────────────────
combined_pub_enroll = pd.concat([enroll_18, enroll_19, enroll_20])
cols = list(combined_pub_enroll.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_enroll = combined_pub_enroll[cols]

print('Shape:', combined_pub_enroll.shape)
combined_pub_enroll.head()

Shape: (84, 53)


,Year,Grade_1_Enrollment_Total,Grade_1_Enrollment_Girl,Grade_1_Repeaters_Total,Grade_1_Repeaters_Girl,Grade_2_Enrollment_Total,Grade_2_Enrollment_Girl,Grade_2_Repeaters_Total,Grade_2_Repeaters_Girl,Grade_3_Enrollment_Total,...,Grade_11_Repeaters_Total,Grade_11_Repeaters_Girl,Grade_12_Enrollment_Total,Grade_12_Enrollment_Girl,Grade_12_Repeaters_Total,Grade_12_Repeaters_Girl,Total_Enrollment_Total,Total_Enrollment_Girl,Total_Repeaters_Total,Total_Repeaters_Girl
Province,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,18465,8748,1074,412,17049,8062,817,256,16656,...,24,10,3272,1781,171,74,135966,67887,4051,1371
Battambang,2018,34641,16139,4037,1512,31386,14626,2462,852,29023,...,72,28,5290,3077,248,132,234716,116219,12244,4362
Kampong Cham,2018,27237,12570,3946,1498,25534,11826,2868,1055,24421,...,40,9,6228,3458,329,149,206003,102607,14544,5235
Kampong Chhnang,2018,13264,6211,1395,502,12580,5995,1002,307,12692,...,21,7,3461,1917,179,88,111230,56189,5080,1676
Kampong Speu,2018,19320,9030,1679,647,18429,8671,1210,440,18766,...,9,0,4125,2092,76,31,160336,78979,5931,2157


## 3. Teaching Staff by Education Level

All three years share the same 19-column layout:  
`Province` | Teaching (Primary, L.Sec, U.Sec, Graduate, PostGrad, PhD) | Non-Teaching (same 6) | No-Ped-Training (same 6)

In [4]:
staff_cols = [
    'Province',
    'T_Primary', 'T_LSec', 'T_USec', 'T_Graduate', 'T_PostGrad', 'T_PhD',
    'NT_Primary', 'NT_LSec', 'NT_USec', 'NT_Graduate', 'NT_PostGrad', 'NT_PhD',
    'NoPed_Primary', 'NoPed_LSec', 'NoPed_USec', 'NoPed_Graduate', 'NoPed_PostGrad', 'NoPed_PhD',
]

def clean_pub_staff(df):
    # Find first row where column-1 is numeric (= first province data row)
    start = next(
        i for i, v in enumerate(df.iloc[:, 1])
        if pd.notna(pd.to_numeric(v, errors='coerce'))
    )
    data = df.iloc[start:].copy().reset_index(drop=True)
    data = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    data.columns = staff_cols
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce').astype('Int64')

staff_sheet_names = {
    2018: 'School Staff by Education Leve',   # truncated tab name in 2018 file
    2019: 'School Staff by Education Level',
    2020: 'School Staff by Education Level',
}
fnames = {
    2018: 'organized_education_data/public_schools_2018-2019.xlsx',
    2019: 'organized_education_data/public_schools_2019-2020.xlsx',
    2020: 'organized_education_data/public_schools_2020-2021.xlsx',
}

staff_frames = []
for year in [2018, 2019, 2020]:
    raw = pd.read_excel(fnames[year], sheet_name=staff_sheet_names[year], header=None)
    df  = clean_pub_staff(raw)
    df['Year'] = year
    staff_frames.append(df)

combined_pub_staff = pd.concat(staff_frames)
cols = list(combined_pub_staff.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_staff = combined_pub_staff[cols]

print('Shape:', combined_pub_staff.shape)
combined_pub_staff.head()

Shape: (84, 19)


,Year,T_Primary,T_LSec,T_USec,T_Graduate,T_PostGrad,T_PhD,NT_Primary,NT_LSec,NT_USec,NT_Graduate,NT_PostGrad,NT_PhD,NoPed_Primary,NoPed_LSec,NoPed_USec,NoPed_Graduate,NoPed_PostGrad,NoPed_PhD
Province,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,49,779,2697,1188,51,0,14,204,393,232,42,0,0,0,10,1,0,0
Battambang,2018,79,1132,3450,2146,145,5,41,352,735,599,78,3,2,13,22,10,0,0
Kampong Cham,2018,127,1713,3043,914,38,0,59,531,893,186,29,0,6,17,34,5,0,0
Kampong Chhnang,2018,19,595,1912,932,40,0,10,158,341,169,22,0,0,1,1,4,0,0
Kampong Speu,2018,53,921,2174,1276,47,0,10,187,286,224,33,2,0,4,1,2,0,0


## 4. Student Flow Rates — Dropout / Promotion / Repetition

- **2018-19**: three sheets (Gr1-4, Gr5-8, Gr9-12) — these contain **2017-18** flow rates
- **2019-20 & 2020-21**: one sheet each, all 12 grades

Values are percentages (0–100).

In [5]:
def parse_flow_sheet(fname, sheet, g_start, g_end):
    df    = pd.read_excel(fname, sheet_name=sheet, header=None)
    start = next(
        i for i, v in enumerate(df.iloc[:, 0])
        if isinstance(v, str) and v not in ('Province',) and not v.startswith('Table')
    )
    data = df.iloc[start:].copy().reset_index(drop=True)
    data = data[data.iloc[:, 0].notna()].reset_index(drop=True)
    n    = g_end - g_start + 1
    col_names = ['Province']
    for g in range(g_start, g_end + 1):
        col_names += [f'Grade_{g}_Promotion', f'Grade_{g}_Repetition', f'Grade_{g}_Dropout']
    data = data.iloc[:, :1 + n * 3].copy()
    data.columns = col_names
    data = data.set_index('Province')
    return data.apply(pd.to_numeric, errors='coerce')


# 2018-19: three separate sheets
f18 = 'organized_education_data/public_schools_2018-2019.xlsx'
flow_18 = pd.concat([
    parse_flow_sheet(f18, 'Flow Rates Both Gr1-4 2017-18',  1,  4),
    parse_flow_sheet(f18, 'Flow Rates Both Gr5-8 2017-18',  5,  8),
    parse_flow_sheet(f18, 'Flow Rates Both Gr9-12 2017-18', 9, 12),
], axis=1)
flow_18['Year'] = 2018

# 2019-20: single sheet (all grades)
flow_19 = parse_flow_sheet(
    'organized_education_data/public_schools_2019-2020.xlsx',
    'Student Flow Rates (Both) 2018-', 1, 12)
flow_19['Year'] = 2019

# 2020-21: single sheet (all grades)
flow_20 = parse_flow_sheet(
    'organized_education_data/public_schools_2020-2021.xlsx',
    'Student Flow Rates (Both) 2020-', 1, 12)
flow_20['Year'] = 2020

combined_pub_flow = pd.concat([flow_18, flow_19, flow_20])
cols = list(combined_pub_flow.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_pub_flow = combined_pub_flow[cols]

print('Shape:', combined_pub_flow.shape)
combined_pub_flow.head()

Shape: (84, 37)


,Year,Grade_1_Promotion,Grade_1_Repetition,Grade_1_Dropout,Grade_2_Promotion,Grade_2_Repetition,Grade_2_Dropout,Grade_3_Promotion,Grade_3_Repetition,Grade_3_Dropout,...,Grade_9_Dropout,Grade_10_Promotion,Grade_10_Repetition,Grade_10_Dropout,Grade_11_Promotion,Grade_11_Repetition,Grade_11_Dropout,Grade_12_Promotion,Grade_12_Repetition,Grade_12_Dropout
Province,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,90.7,6.0,3.3,91.6,4.7,3.7,94.7,3.4,2.0,...,18.4,85.6,0.6,13.8,89.7,0.7,9.6,54.9,5.5,39.7
Battambang,2018,83.8,11.7,4.5,88.5,8.0,3.6,88.6,5.7,5.7,...,17.0,82.4,1.2,16.4,87.5,1.2,11.3,66.1,4.6,29.3
Kampong Cham,2018,81.6,14.2,4.2,84.4,10.9,4.7,86.5,8.9,4.6,...,17.2,80.0,2.0,18.1,89.9,0.6,9.5,56.6,5.3,38.1
Kampong Chhnang,2018,88.0,10.6,1.4,90.3,7.6,2.0,91.5,6.3,2.2,...,15.7,81.9,0.8,17.3,89.2,0.6,10.2,60.2,5.2,34.6
Kampong Speu,2018,89.3,8.7,2.0,92.5,6.3,1.2,92.7,4.5,2.8,...,22.7,79.4,0.3,20.3,89.4,0.2,10.4,84.4,1.9,13.7


## 5. Sanity Check — Whole Kingdom totals

In [6]:
wk_main = combined_pub_main[
    combined_pub_main.index == 'Whole Kingdom'].sort_values('Year')
print('=== Main (Whole Kingdom) ===')
print(wk_main[[
    'Year', 'Schools', 'Classes',
    'Enrollment_Total', 'Repeaters_Total',
    'TeachStaff_Total', 'NonTeachStaff_Total'
]].to_string())

wk_enroll = combined_pub_enroll[
    combined_pub_enroll.index == 'Whole Kingdom'].sort_values('Year')
print('\n=== Enroll totals (Whole Kingdom) ===')
print(wk_enroll[[
    'Year',
    'Total_Enrollment_Total', 'Total_Enrollment_Girl',
    'Total_Repeaters_Total',  'Total_Repeaters_Girl'
]].to_string())

wk_staff = combined_pub_staff[
    combined_pub_staff.index == 'Whole Kingdom'].sort_values('Year')
print('\n=== Staff education level (Whole Kingdom) ===')
print(wk_staff[[
    'Year', 'T_Primary', 'T_LSec', 'T_USec', 'T_Graduate', 'T_PostGrad', 'T_PhD'
]].to_string())

wk_flow = combined_pub_flow[
    combined_pub_flow.index == 'Whole Kingdom'].sort_values('Year')
dropout_cols = [f'Grade_{g}_Dropout' for g in range(1, 13)]
print('\n=== Dropout % by grade (Whole Kingdom) ===')
print(wk_flow[['Year'] + dropout_cols].to_string())

=== Main (Whole Kingdom) ===
               Year  Schools  Classes  Enrollment_Total  Repeaters_Total  TeachStaff_Total  NonTeachStaff_Total
Province                                                                                                       
Whole Kingdom  2018    13300    89897           3189172           148021             93703                19066
Whole Kingdom  2019    13482    91259           3210285           150757             93225                19862
Whole Kingdom  2020    13681    95909           3277076           175718             94718                20831

=== Enroll totals (Whole Kingdom) ===
               Year  Total_Enrollment_Total  Total_Enrollment_Girl  Total_Repeaters_Total  Total_Repeaters_Girl
Province                                                                                                       
Whole Kingdom  2018                 2971663                1469191                 148021                 52937
Whole Kingdom  2019                 